In [9]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import cv2
from keras.utils import img_to_array
IMG_SIZE = 128
DATA_DIR = "/kaggle/input/datasets/jangedoo/utkface-new/UTKFace"

def load_data(data_dir):
    image_arr, ages = [], []
    for image in os.listdir(data_dir):
        age = int(image.split("_")[0])
        img = cv2.imread(f'{data_dir}/{image}', cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (48, 48))
        img_arr = img_to_array(img)
        
        image_arr.append(img_arr)
        ages.append(age)
    return image_arr, np.array(ages, dtype=np.float32)

image_arr, ages = load_data(DATA_DIR)
print(f"Total samples: {len(image_arr)}")

Total samples: 23708


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    image_arr, ages, test_size=0.2, random_state=42
)
# X_train, val_paths, train_ages, val_ages = train_test_split(
#     train_paths, train_ages, test_size=0.15, random_state=42
# )

# print(f"Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

In [11]:

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1/255, width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True)
train_data = train_datagen.flow(x=np.asarray(X_train), y=y_train, batch_size=32)

test_datagen = ImageDataGenerator(rescale=1/255)
test_data = test_datagen.flow(np.asarray(X_test), y_test, batch_size=32)

In [12]:
from keras.layers import *
from keras import regularizers

l2 = regularizers.l2(0.00015)
model = tf.keras.Sequential([
    
    tf.keras.Input(shape=(48, 48, 1)),
    
    Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same', kernel_regularizer=l2),
    BatchNormalization(),
    
    Conv2D(32, kernel_size = (3,3), activation = 'relu', padding='same', kernel_regularizer=l2),
    BatchNormalization(),

    MaxPooling2D(),
    Dropout(0.3),

    Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same', kernel_regularizer=l2),
    BatchNormalization(),
    
    Conv2D(64, kernel_size = (3,3), activation = 'relu', padding='same', kernel_regularizer=l2),
    BatchNormalization(),

    MaxPooling2D(),
    Dropout(0.3),

    Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same', kernel_regularizer=l2),
    BatchNormalization(),
    
    Conv2D(256, kernel_size = (3,3), activation = 'relu', padding='same', kernel_regularizer=l2),
    BatchNormalization(),

    MaxPooling2D(),
    Dropout(0.3),

    Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same', kernel_regularizer=l2),
    BatchNormalization(),
    
    Conv2D(512, kernel_size = (3,3), activation = 'relu', padding='same', kernel_regularizer=l2),
    BatchNormalization(),

    MaxPooling2D(),
    
    GlobalAveragePooling2D(),

    Dense(256, activation='relu', kernel_regularizer=regularizers.L2(0.0005)),
    Dropout(0.3),

    Dense(1, activation='relu')])

In [13]:
model.compile(loss='mean_squared_error',
              optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              metrics=['mae'])

In [14]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_mae", factor=0.5, patience=6, min_lr=1e-6),
    tf.keras.callbacks.EarlyStopping(monitor="val_mae", patience=12, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("best_age_model.keras", monitor="val_mae", save_best_only=True)
]

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=60,
    callbacks=callbacks
)

Epoch 1/60
593/593 ━━━━━━━━━━━━━━━━━━━━ 35s 36ms/step - loss: 251.7534 - mae: 11.8706 - val_loss: 147.1528 - val_mae: 9.0929 - learning_rate: 0.0010
Epoch 2/60
593/593 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 157.1400 - mae: 9.2258 - val_loss: 176.6570 - val_mae: 9.9470 - learning_rate: 0.0010
Epoch 3/60
593/593 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 133.6951 - mae: 8.4683 - val_loss: 172.9967 - val_mae: 9.6403 - learning_rate: 0.0010
Epoch 4/60
593/593 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 120.2975 - mae: 8.0015 - val_loss: 106.6441 - val_mae: 7.7251 - learning_rate: 0.0010
Epoch 5/60
593/593 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 113.9368 - mae: 7.7694 - val_loss: 106.4367 - val_mae: 7.3464 - learning_rate: 0.0010
Epoch 6/60
593/593 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - loss: 107.2298 - mae: 7.6021 - val_loss: 114.6255 - val_mae: 7.7217 - learning_rate: 0.0010
Epoch 7/60
593/593 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 98.9199 - mae: 7.2819 - val_loss: 87.5490 - val_m

In [15]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 48, 48, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 6, 6, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 6, 6, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 6, 6, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 7,018,277 (26.77 MB)

 Trainable params: 2,338,529 (8.92 MB)

 Non-trainable params: 2,688 (10.50 KB)

 Optimizer params: 4,677,060 (17.84 MB)

In [16]:
test_loss, test_mae = model.evaluate(test_data)
print(f"Test MSE: {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f} years")

149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 59.8978 - mae: 5.1629
Test MSE: 59.8978
Test MAE: 5.1629 years
